In [ ]:
import os
import sys
import json
import csv
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
import geopandas as gpd
import h3
import h3pandas

from shapely import wkt
from shapely.geometry import shape
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# Get the parent directory of the current directory (which is 'notebooks')
# and add it to the system path
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can use absolute imports from the project root
from utils.geometry import get_bearing, get_bearing_label, convert_geometry

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Geo Admin boundaries

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")

In [ ]:
# coasts_gdf = gpd.read_file('natural_earth/ne_10m_coastline.shp')
# admin0_gdf = gpd.read_file('natural_earth/ne_10m_admin_0_boundary_lines_land.shp')
# island_gdf = gpd.read_file('natural_earth/ne_10m_minor_islands_coastline.shp')
# seams_gdf = gpd.read_file('natural_earth/ne_10m_admin_1_seams.shp')

In [ ]:
data_fn = os.path.join(DATA_DIR, 'natural_earth\\ne_10m_admin_0_countries_usa.shp')

In [ ]:
world_gdf = gpd.read_file(data_fn)

In [ ]:
world_gdf.plot(figsize=(13,8))

#### Create a 10km buffer for coastlines

In [ ]:
country_gdf = world_gdf[world_gdf['SOV_A3'] == 'USA']

In [ ]:
country_gdf = country_gdf.to_crs(3857)

In [ ]:
buffered_coasts = country_gdf.buffer(10000)

In [ ]:
# buffered_gdf = gpd.GeoDataFrame(geometry=buffered_coasts, crs='EPSG:3857')

In [ ]:
buffered_gdf = buffered_gdf.to_crs(4326)

In [ ]:
# buffered_gdf.plot(figsize=(13, 8))

### H3

In [ ]:
# # Merge all GeoDataFrames
# world_gdf = pd.concat([coasts_gdf, island_gdf], axis=0)
# world_gdf = pd.concat([world_gdf, seams_gdf], axis=0)

In [ ]:
# len(coasts_gdf) + len(island_gdf) + len(seams_gdf)

In [ ]:
limit_cols = []
for col in world_gdf.columns:
    null_df = world_gdf[world_gdf[col].isna()]
    if col.startswith('NAME') or col.startswith('ADM0') or col.startswith('FCLASS') or col.startswith('BRK') or col.startswith('WOE') or col.startswith('WB') or col.startswith('ISO_N3') or col.startswith('ISO_A2_EH'):
        pass
    elif 'LABEL' in col or 'ZOOM' in col or 'MAPCOLOR' in col:
        pass
    elif col in ['FIPS_10', 'FIPS_10', 'ISO_A3_EH', 'INCOME_GRP', 'SUBUNIT', 'FORMAL_EN', 'GEOUNIT', 'GU_A3', 'SU_A3', 'UN_A3', 'TLC', 'TYPE', 'LEVEL', 'featurecla', 'scalerank', 'GEOU_DIF', 'SU_DIF', 'HOMEPART', 'BRK_DIFF', 'POSTAL', 'ABBREV', 'TINY']:
        pass
    elif null_df.empty:
        limit_cols.append(col)
    elif len(null_df) < 0.1 * len(world_gdf):
        limit_cols.append(col)

In [ ]:
limit_cols

In [ ]:
world_gdf = world_gdf.filter(limit_cols)

In [ ]:
world_gdf

In [ ]:
country_lookup = {}

In [ ]:
grp_by_admin = world_gdf[['SOV_A3', 'WIKIDATAID']].groupby('SOV_A3').agg('count')

In [ ]:
grp_by_admin = grp_by_admin.reset_index()

In [ ]:
# grp_by_admin[grp_by_admin['WIKIDATAID'] > 1]
multi_sov = grp_by_admin[grp_by_admin['WIKIDATAID'] > 1].SOV_A3.to_list()

In [ ]:
len(world_gdf['WIKIDATAID'].drop_duplicates().to_list()) == len(world_gdf)

In [ ]:
for country in world_gdf.SOV_A3.drop_duplicates().to_list():

    tmp_gdf = world_gdf[(world_gdf['SOV_A3'] == country)]
    try:
    
        if country in multi_sov:
            sub_list = tmp_gdf.ADMIN.drop_duplicates().to_list()
            for sub in sub_list:
                tmp_tmp_gdf = tmp_gdf[tmp_gdf['ADMIN'] == sub]
                tmp_hex_df = tmp_tmp_gdf.h3.polyfill(5, explode=True)
                subname = sub.lower().replace(' ', '_')
                country_key = f"{country}_{subname}"
                country_lookup[country_key] = tmp_hex_df.h3_polyfill.drop_duplicates().to_list()
        else:
            tmp_hex_df = tmp_gdf.h3.polyfill(5, explode=True)
            country_lookup[country] = tmp_hex_df.h3_polyfill.drop_duplicates().to_list()
    except:
        country_lookup[country] = []


In [ ]:
country_verbose_lookup = {}
for country in world_gdf.ADMIN.drop_duplicates().to_list():

    tmp_gdf = world_gdf[(world_gdf['ADMIN'] == country)]
    try:
        tmp_hex_df = tmp_gdf.h3.polyfill(5, explode=True)
        country_verbose_lookup[country] = tmp_hex_df.h3_polyfill.drop_duplicates().to_list()
    except:
        country_verbose_lookup[country] = []


In [ ]:
len(country_lookup)

In [ ]:
len(country_verbose_lookup)

In [ ]:
with open(os.path.join(DATA_DIR, 'opto_index\\country_code_3_h5_ids.json'), 'w') as fw:
    json_data = json.dumps(country_lookup)
    fw.write(json_data)

In [ ]:
with open(os.path.join(DATA_DIR, 'opto_index\\country_name_h5_ids.json'), 'w') as fw:
    json_data = json.dumps(country_verbose_lookup)
    fw.write(json_data)